# *Getting and Understanding the Datasets*

In [1]:
import pandas as pd

# CICIDS2017
cic = pd.read_csv("../data/raw/cicids2017/Wednesday-workingHours.pcap_ISCX.csv")
cic.columns = cic.columns.str.strip() # fixes the leading-space column name issue
print("CICIDS2017 shape",cic.shape)
print(cic.columns.tolist()) 
print(cic['Label'].value_counts())

# UNSW-NB15
unsw_train = pd.read_csv("../data/raw/unsw-nb15/UNSW_NB15_training-set.csv")
print("UNSW_NB15 training shape", unsw_train.shape)
#print(unsw_train.columns.tolist()) 
print(unsw_train['label'].value_counts())
print(unsw_train['attack_cat'].value_counts())

CICIDS2017 shape (692703, 79)
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CW

In [2]:
print(unsw_train.shape)
# if this prints 82332, you likely loaded UNSW_NB15_testing-set.csv by mistake

(175341, 45)


# *Prepocessing*

In [ ]:
# CICISD2017 preprocessing
# handling missing valeues and infinite values
import numpy as np
print("NAN count: ", cic.isna().sum().sum())
print("Infinite count: ", np.isinf(cic.select_dtypes(include=[np.number])).sum().sum())

# Replacing the infinites with NaN and then dropping the rows with NaN values
cic.replace([np.inf, -np.inf], np.nan, inplace=True)
cic.dropna(inplace=True)
print("Shape after cleaning: ", cic.shape)
print("\n")

# Remove duplicate rows
dupes = cic.duplicated().sum()
print("Number of duplicate rows: ", dupes)
cic.drop_duplicates(inplace=True)
print("Shape after removing duplicates: ", cic.shape)
print("\n")

# Binary encoding: 0 = benign, 1 = any attack
cic['Label_binary'] = cic['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
print(cic['Label_binary'].value_counts())
print("\n")

# Seperate features(X) and target(y)
#Drop the 
X = cic.drop(columns=['Label', 'Label_binary'])
y = cic['Label_binary']


print("X shape: ", X.shape)
print("Any non-numeric colums left?", X.select_dtypes(exclude=[np.number]). columns.tolist())
print("\n")

# Normalization(scaling)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled shape:", X_scaled.shape)
print("\n")

NAN count:  1008
Infinite count:  1586
Shape after cleaning:  (691406, 79)


Number of duplicate rows:  80914
Shape after removing duplicates:  (610492, 79)


Label_binary
0    416736
1    193756
Name: count, dtype: int64


X shape:  (610492, 78)
Any non-numeric colums left? []


Scaled shape: (610492, 78)




In [6]:
# UNSW-NB15 train preprocessing

print("NAN count: ", unsw_train.isna().sum().sum())
print("Infinite count: ", np.isinf(unsw_train.select_dtypes(include=[np.number])).sum().sum())

# Remove duplicate rows
dupes_unsw = unsw_train.duplicated().sum()
print("Number of duplicate rows: ", dupes_unsw)
unsw_train.drop_duplicates(inplace=True)
print("Shape after removing duplicates: ", unsw_train.shape)
print("\n")

#Check non-numeric coulmns (UNSW_NB15 likely HAS some- e.g. 'proto, 'service')
print("Any non-numeric colums left?", unsw_train.select_dtypes(exclude=[np.number]). columns.tolist())

NAN count:  0
Infinite count:  0
Number of duplicate rows:  0
Shape after removing duplicates:  (175341, 45)


Any non-numeric colums left? ['proto', 'service', 'state', 'attack_cat']


In [7]:
# checking cardinality for UNSW-NB15
for col in ['proto', 'service', 'state']:
    print(col, ":", unsw_train[col].nunique(), "unique values")

#keeping the attack_cat aside
y_unsw_binary = unsw_train['label']
y_unsw_multiclass = unsw_train['attack_cat']

X_unsw = unsw_train.drop(columns=['label', 'attack_cat'])
print(X_unsw.shape)
print(X_unsw.select_dtypes(exclude=[np.number]).columns.tolist())  # should now only show proto, service, state

proto : 133 unique values
service : 13 unique values
state : 9 unique values
(175341, 43)
['proto', 'service', 'state']


In [9]:
#check how skewed proto actually is
print(unsw_train['proto'].value_counts().head(15))
print("\nprotocols appearing in <0.1% of rows:", (unsw_train['proto'].value_counts(normalize=True) < 0.001).sum())

#group rare protocols into 'other' (threshold: appears in <0.1% of rows)
proto_counts = unsw_train['proto'].value_counts(normalize=True)
rare_proto = proto_counts[proto_counts < 0.001].index
unsw_train['proto_grouped'] = unsw_train['proto'].apply(lambda x: 'other' if x in rare_proto else x)
print(unsw_train['proto_grouped'].nunique(), "categories after grouping")

# one-hot encoding for categorical features
X_unsw_encoded = pd.get_dummies(X_unsw.assign(proto=unsw_train['proto_grouped']),
                                columns=['proto', 'service', 'state'],
                                drop_first=False
)
print("Shape after encoding:", X_unsw_encoded.shape)

proto
tcp       79946
udp       63283
unas      12084
arp        2859
ospf       2595
sctp       1150
any         300
gre         225
sun-nd      201
swipe       201
mobile      201
pim         201
ipv6        201
rsvp        200
sep         193
Name: count, dtype: int64

protocols appearing in <0.1% of rows: 118
16 categories after grouping
Shape after encoding: (175341, 78)


In [10]:
# Preprocessing for UNSW-NB15 test set

#loading the files
unsw_test = pd.read_csv("../data/raw/unsw-nb15/UNSW_NB15_testing-set.csv")
print("UNSW_NB15 testing shape", unsw_test.shape)

# check NaN/Inf/duplicates(same as training):
print("NAN count: ", unsw_test.isna().sum().sum())
print("Infinite count: ", np.isinf(unsw_test.select_dtypes(include=[np.number])).sum().sum())

# Remove duplicate rows
dupes_test = unsw_test.duplicated().sum()
print("Number of duplicate rows: ", dupes_test)
unsw_test.drop_duplicates(inplace=True)
print("Shape after removing duplicates: ", unsw_test.shape)
print("\n")

# Grouping proto
unsw_test['proto_grouped'] = unsw_test['proto'].apply(lambda x: 'other' if x in rare_proto else x)
print(unsw_test['proto_grouped'].nunique(), "categories after grouping")

#Seperate labels
y_unsw_test_binary = unsw_test['label']
y_unsw_test_multiclass = unsw_test['attack_cat']

X_unsw_test = unsw_test.drop(columns=['label', 'attack_cat'])

# One-hot encoding
X_unsw_test_encoded = pd.get_dummies(X_unsw_test.assign(proto=unsw_test['proto_grouped']),
                                columns=['proto', 'service', 'state'],
                                drop_first=False
)
X_unsw_test_encoded = X_unsw_test_encoded.reindex(columns=X_unsw_encoded.columns, fill_value=0)

print("Test shape:", X_unsw_test_encoded.shape)
#print("Train shape:", X_unsw_encoded.shape)
print("Columns match:", list(X_unsw_test_encoded.columns) == list(X_unsw_encoded.columns))

UNSW_NB15 testing shape (82332, 45)
NAN count:  0
Infinite count:  0
Number of duplicate rows:  0
Shape after removing duplicates:  (82332, 45)


16 categories after grouping
Test shape: (82332, 78)
Columns match: True


In [12]:
cic.to_csv("../data/processed/cicids2017_wednesday_clean.csv", index=False)
X_unsw_encoded.to_csv("../data/processed/unsw_train_encoded.csv", index=False)
y_unsw_binary.to_csv("../data/processed/unsw_train_labels.csv", index=False)